# NARX V5

V5 chuyển khỏi MLP sang nonlinear tree ensemble cho tabular lag features:

- Model: `HistGradientBoostingRegressor`
- Search nhiều order trọng điểm, không giả định ARX best là NARX best
- 16 augmented features
- z-score target/input để giữ cùng scale pipeline
- clip free-run Q1%-Q99% train
- chọn theo validation `FIT_sim`

Mục tiêu đầu tiên: vượt NARX MLP tốt nhất. Mục tiêu cao hơn: tiến gần hoặc vượt ARX best.


## 1. Import và cấu hình


In [1]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

WORK_DIR = Path.cwd()
PROJECT_ROOT = WORK_DIR.parent if WORK_DIR.name == "NARX" else WORK_DIR
OUT_DIR = PROJECT_ROOT / "NARX"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from arx_pipeline import DataConfig, SplitConfig, load_or_generate_data, split_time_series
from narx_pipeline import (
    NarxConfig,
    build_augmented_df,
    fit_zscore_stats,
    apply_zscore,
    inverse_zscore_y,
    scaled_clip_bounds,
    build_narx_matrix,
    simulate_narx,
    compute_metrics,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

ORDER_CANDIDATES = [
    (1, 1, 2),
    (1, 2, 2),
    (1, 3, 2),
    (1, 5, 2),
    (2, 1, 2),
    (2, 3, 2),
    (2, 5, 2),
    (3, 1, 2),
    (5, 1, 2),
    (5, 3, 2),
]

MODEL_CANDIDATES = [
    {"max_iter": 120, "learning_rate": 0.05, "max_leaf_nodes": 15, "l2_regularization": 0.0},
    {"max_iter": 180, "learning_rate": 0.05, "max_leaf_nodes": 31, "l2_regularization": 0.0},
    {"max_iter": 120, "learning_rate": 0.10, "max_leaf_nodes": 15, "l2_regularization": 1e-4},
    {"max_iter": 180, "learning_rate": 0.10, "max_leaf_nodes": 31, "l2_regularization": 1e-4},
]

CLIP_QUANTILES = (0.01, 0.99)
RANDOM_STATE = 42
TOP_K_TEST = 8


## 2. Data và preprocessing


In [2]:
DATA_CONFIG = DataConfig(
    csv_path=PROJECT_ROOT / "greenhouse_data.csv",
    generator_script_path=PROJECT_ROOT / "data_generator.py",
    force_regenerate_from_script=False,
    auto_save_generated_csv=True,
)
SPLIT_CONFIG = SplitConfig(train_ratio=0.60, val_ratio=0.20)

BASELINE_INPUT_COLS = ("Temperature", "Humidity", "Light", "Drip", "Mist", "Fan")
AUGMENTED_INPUT_COLS = (
    *BASELINE_INPUT_COLS,
    "Light_log",
    "Temp_x_Humi",
    "Temp_x_Light",
    "Humi_x_Light",
    "SP_Center",
    "SP_Width",
    "Month_sin",
    "Month_cos",
    "Season_sin",
    "Season_cos",
)

df_full, true_params, data_source = load_or_generate_data(DATA_CONFIG)
df_aug = build_augmented_df(df_full)
df_train, df_val, df_test = split_time_series(df_aug, SPLIT_CONFIG)
SCALE_COLS = ("Soil_Moisture", *AUGMENTED_INPUT_COLS)
scale_stats = fit_zscore_stats(df_train, SCALE_COLS)
clip_bounds_real, clip_bounds_scaled = scaled_clip_bounds(df_train, scale_stats, CLIP_QUANTILES)

df_train_z = apply_zscore(df_train, scale_stats)
df_val_z = apply_zscore(df_val, scale_stats)
df_test_z = apply_zscore(df_test, scale_stats)

pd.Series({
    "data_source": data_source,
    "train_rows": len(df_train),
    "val_rows": len(df_val),
    "test_rows": len(df_test),
    "clip_real": clip_bounds_real,
    "clip_scaled": clip_bounds_scaled,
})


data_source                      CSV:greenhouse_data.csv
train_rows                                         63072
val_rows                                           21024
test_rows                                          21024
clip_real          (50.5465794049447, 64.61272806162447)
clip_scaled    (-2.0642032653307445, 2.1303300718324136)
dtype: object

## 3. Helper train/evaluate


In [3]:
def make_model(params: dict) -> HistGradientBoostingRegressor:
    return HistGradientBoostingRegressor(
        loss="squared_error",
        max_iter=int(params["max_iter"]),
        learning_rate=float(params["learning_rate"]),
        max_leaf_nodes=int(params["max_leaf_nodes"]),
        l2_regularization=float(params["l2_regularization"]),
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=RANDOM_STATE,
    )


def one_step_metrics(model, x_mat: np.ndarray, y_vec: np.ndarray) -> dict[str, float]:
    pred = model.predict(x_mat)
    return compute_metrics(
        inverse_zscore_y(y_vec, scale_stats),
        inverse_zscore_y(pred, scale_stats),
        x_mat.shape[1],
    )


def sim_metrics(model, df_z: pd.DataFrame, config: NarxConfig) -> dict[str, float]:
    y_pred, y_true = simulate_narx(df_z, model, config)
    return compute_metrics(
        inverse_zscore_y(y_true, scale_stats),
        inverse_zscore_y(y_pred, scale_stats),
        config.n_features,
    )


def train_candidate(order: tuple[int, int, int], params: dict) -> tuple[HistGradientBoostingRegressor, dict]:
    na, nb, nk = order
    config = NarxConfig(
        na=na,
        nb=nb,
        nk=nk,
        input_cols=AUGMENTED_INPUT_COLS,
        simulation_clip=clip_bounds_scaled,
    )
    x_train, y_train = build_narx_matrix(df_train_z, config)
    x_val, y_val = build_narx_matrix(df_val_z, config)
    model = make_model(params)
    start = time.time()
    model.fit(x_train, y_train)
    train_seconds = time.time() - start
    val_1 = one_step_metrics(model, x_val, y_val)
    val_sim = sim_metrics(model, df_val_z, config)
    return model, {
        "order": f"({na},{nb},{nk})",
        "na": na,
        "nb": nb,
        "nk": nk,
        "n_features": config.n_features,
        "max_iter": int(params["max_iter"]),
        "learning_rate": float(params["learning_rate"]),
        "max_leaf_nodes": int(params["max_leaf_nodes"]),
        "l2_regularization": float(params["l2_regularization"]),
        "n_iter": int(model.n_iter_),
        "train_seconds": float(train_seconds),
        "val_FIT_1step": val_1["FIT"],
        "val_RMSE_1step": val_1["RMSE"],
        "val_FIT_sim": val_sim["FIT"],
        "val_RMSE_sim": val_sim["RMSE"],
    }


def test_candidate(model, order: tuple[int, int, int]) -> dict[str, float]:
    na, nb, nk = order
    config = NarxConfig(
        na=na,
        nb=nb,
        nk=nk,
        input_cols=AUGMENTED_INPUT_COLS,
        simulation_clip=clip_bounds_scaled,
    )
    x_test, y_test = build_narx_matrix(df_test_z, config)
    test_1 = one_step_metrics(model, x_test, y_test)
    test_sim = sim_metrics(model, df_test_z, config)
    return {
        "test_FIT_1step": test_1["FIT"],
        "test_RMSE_1step": test_1["RMSE"],
        "test_FIT_sim": test_sim["FIT"],
        "test_RMSE_sim": test_sim["RMSE"],
        "test_Bias_sim": test_sim["Bias"],
    }


## 4. Search validation


In [4]:
rows = []
models = {}
orders_by_id = {}
params_by_id = {}
candidate_id = 0
total = len(ORDER_CANDIDATES) * len(MODEL_CANDIDATES)

for order in ORDER_CANDIDATES:
    for params in MODEL_CANDIDATES:
        candidate_id += 1
        print(f"{candidate_id}/{total}: order={order}, params={params}")
        model, row = train_candidate(order, params)
        row["candidate_id"] = candidate_id
        rows.append(row)
        models[candidate_id] = model
        orders_by_id[candidate_id] = order
        params_by_id[candidate_id] = params

search_df = pd.DataFrame(rows).sort_values(
    ["val_FIT_sim", "val_FIT_1step"],
    ascending=[False, False],
).reset_index(drop=True)

search_df.head(20).round(4)


1/40: order=(1, 1, 2), params={'max_iter': 120, 'learning_rate': 0.05, 'max_leaf_nodes': 15, 'l2_regularization': 0.0}


2/40: order=(1, 1, 2), params={'max_iter': 180, 'learning_rate': 0.05, 'max_leaf_nodes': 31, 'l2_regularization': 0.0}


3/40: order=(1, 1, 2), params={'max_iter': 120, 'learning_rate': 0.1, 'max_leaf_nodes': 15, 'l2_regularization': 0.0001}


4/40: order=(1, 1, 2), params={'max_iter': 180, 'learning_rate': 0.1, 'max_leaf_nodes': 31, 'l2_regularization': 0.0001}


5/40: order=(1, 2, 2), params={'max_iter': 120, 'learning_rate': 0.05, 'max_leaf_nodes': 15, 'l2_regularization': 0.0}


...

37/40: order=(5, 3, 2), params={'max_iter': 120, 'learning_rate': 0.05, 'max_leaf_nodes': 15, 'l2_regularization': 0.0}


38/40: order=(5, 3, 2), params={'max_iter': 180, 'learning_rate': 0.05, 'max_leaf_nodes': 31, 'l2_regularization': 0.0}


39/40: order=(5, 3, 2), params={'max_iter': 120, 'learning_rate': 0.1, 'max_leaf_nodes': 15, 'l2_regularization': 0.0001}


40/40: order=(5, 3, 2), params={'max_iter': 180, 'learning_rate': 0.1, 'max_leaf_nodes': 31, 'l2_regularization': 0.0001}


,order,na,nb,nk,n_features,max_iter,learning_rate,max_leaf_nodes,l2_regularization,n_iter,train_seconds,val_FIT_1step,val_RMSE_1step,val_FIT_sim,val_RMSE_sim,candidate_id
0,"(2,1,2)",2,1,2,18,180,0.05,31,0.0000,180,1.0417,88.0996,0.3573,70.2696,0.8925,18
1,"(2,5,2)",2,5,2,82,180,0.10,31,0.0001,134,1.3407,89.2229,0.3236,70.2502,0.8932,28
2,"(3,1,2)",3,1,2,19,180,0.05,31,0.0000,180,1.3331,89.1890,0.3246,70.1837,0.8951,30
3,"(2,5,2)",2,5,2,82,180,0.05,31,0.0000,180,2.0489,89.2118,0.3239,70.1193,0.8971,26
4,"(2,3,2)",2,3,2,50,180,0.05,31,0.0000,180,1.4675,88.1453,0.3559,70.0739,0.8985,22
5,"(1,3,2)",1,3,2,49,180,0.05,31,0.0000,180,1.9252,87.9795,0.3609,70.0679,0.8986,10
6,"(2,3,2)",2,3,2,50,180,0.10,31,0.0001,180,1.1711,88.4355,0.3472,70.0205,0.9001,24
7,"(3,1,2)",3,1,2,19,180,0.10,31,0.0001,180,0.6797,89.4850,0.3157,69.9887,0.9010,32
8,"(2,3,2)",2,3,2,50,120,0.10,15,0.0001,120,0.6499,88.2212,0.3536,69.9537,0.9021,23
9,"(1,2,2)",1,2,2,33,180,0.05,31,0.0000,180,0.6867,87.9302,0.3624,69.9502,0.9022,6


## 5. Test top candidates


In [5]:
test_rows = []
for _, row in search_df.head(TOP_K_TEST).iterrows():
    cid = int(row["candidate_id"])
    model = models[cid]
    order = orders_by_id[cid]
    test_payload = test_candidate(model, order)
    merged = row.to_dict()
    merged.update(test_payload)
    test_rows.append(merged)

tested_df = pd.DataFrame(test_rows).sort_values(
    ["val_FIT_sim", "test_FIT_sim"],
    ascending=[False, False],
).reset_index(drop=True)

best_by_val = tested_df.iloc[0].to_dict()
best_by_test = tested_df.sort_values(["test_FIT_sim", "val_FIT_sim"], ascending=[False, False]).iloc[0].to_dict()

display(tested_df.round(4))
print("Best by validation:", best_by_val["order"], best_by_val["val_FIT_sim"], best_by_val["test_FIT_sim"])
print("Best by test among tested:", best_by_test["order"], best_by_test["val_FIT_sim"], best_by_test["test_FIT_sim"])


,order,na,nb,nk,n_features,max_iter,learning_rate,max_leaf_nodes,l2_regularization,n_iter,train_seconds,val_FIT_1step,val_RMSE_1step,val_FIT_sim,val_RMSE_sim,candidate_id,test_FIT_1step,test_RMSE_1step,test_FIT_sim,test_RMSE_sim,test_Bias_sim
0,"(2,1,2)",2,1,2,18,180,0.05,31,0.0000,180,1.0417,88.0996,0.3573,70.2696,0.8925,18,87.9859,0.3499,67.4494,0.9481,-0.1533
1,"(2,5,2)",2,5,2,82,180,0.10,31,0.0001,134,1.3407,89.2229,0.3236,70.2502,0.8932,28,88.6720,0.3300,64.1840,1.0433,-0.3028
2,"(3,1,2)",3,1,2,19,180,0.05,31,0.0000,180,1.3331,89.1890,0.3246,70.1837,0.8951,30,89.2627,0.3127,68.5310,0.9166,0.0167
3,"(2,5,2)",2,5,2,82,180,0.05,31,0.0000,180,2.0489,89.2118,0.3239,70.1193,0.8971,26,88.6806,0.3297,66.8571,0.9654,-0.1961
4,"(2,3,2)",2,3,2,50,180,0.05,31,0.0000,180,1.4675,88.1453,0.3559,70.0739,0.8985,22,87.9528,0.3509,65.5465,1.0036,-0.2184
5,"(1,3,2)",1,3,2,49,180,0.05,31,0.0000,180,1.9252,87.9795,0.3609,70.0679,0.8986,10,87.8115,0.3550,67.1410,0.9571,-0.1143
6,"(2,3,2)",2,3,2,50,180,0.10,31,0.0001,180,1.1711,88.4355,0.3472,70.0205,0.9001,24,88.2172,0.3432,68.2020,0.9262,-0.0318
7,"(3,1,2)",3,1,2,19,180,0.10,31,0.0001,180,0.6797,89.4850,0.3157,69.9887,0.9010,32,89.3535,0.3101,68.2828,0.9238,0.0278


Best by validation: (2,1,2) 70.26960183105156 67.44940731208276
Best by test among tested: (3,1,2) 70.18367926430456 68.53103249559682


## 6. So sánh


In [6]:
comparison_rows = []
for label, path in [
    ("ARX Search V1 best", PROJECT_ROOT / "ARX_Model_VersionSearch" / "arx_order_search_v1.json"),
    ("NARX V1", OUT_DIR / "narx_v1.json"),
    ("NARX V2", OUT_DIR / "narx_v2.json"),
    ("NARX V3", OUT_DIR / "narx_v3.json"),
    ("NARX V4", OUT_DIR / "narx_v4.json"),
]:
    if not path.exists():
        continue
    with path.open("r", encoding="utf-8") as f:
        artifact = json.load(f)
    if label.startswith("ARX"):
        best = artifact["best_by_validation"]
        comparison_rows.append({"model": label, "val_FIT_sim": best["val_FIT_sim"], "test_FIT_sim": best["test_FIT_sim"], "test_RMSE_sim": best["test_RMSE_sim"]})
    elif label == "NARX V3":
        best = artifact["best_candidate"]
        comparison_rows.append({"model": label, "val_FIT_sim": best["val_FIT_sim"], "test_FIT_sim": best["test_FIT_sim"], "test_RMSE_sim": best["test_RMSE_sim"]})
    else:
        comparison_rows.append({"model": label, "val_FIT_sim": artifact["metrics"]["validation"]["fit_sim"], "test_FIT_sim": artifact["metrics"]["test"]["fit_sim"], "test_RMSE_sim": artifact["metrics"]["test"]["rmse_sim"]})

comparison_rows.append({
    "model": "NARX V5 HGBR best-by-val",
    "val_FIT_sim": best_by_val["val_FIT_sim"],
    "test_FIT_sim": best_by_val["test_FIT_sim"],
    "test_RMSE_sim": best_by_val["test_RMSE_sim"],
})
comparison_rows.append({
    "model": "NARX V5 HGBR best-tested",
    "val_FIT_sim": best_by_test["val_FIT_sim"],
    "test_FIT_sim": best_by_test["test_FIT_sim"],
    "test_RMSE_sim": best_by_test["test_RMSE_sim"],
})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["test_gain_vs_arx_search_best"] = comparison_df["test_FIT_sim"] - comparison_df.loc[0, "test_FIT_sim"]
comparison_df.round(4)


,model,val_FIT_sim,test_FIT_sim,test_RMSE_sim,test_gain_vs_arx_search_best
0,ARX Search V1 best,68.9559,66.8337,0.9661,0.0000
1,NARX V1,61.2199,54.2491,1.3327,-12.5847
2,NARX V2,65.9420,53.9242,1.3422,-12.9096
3,NARX V3,65.9420,53.9242,1.3422,-12.9096
4,NARX V4,29.6988,-29.2470,3.7649,-96.0807
5,NARX V5 HGBR best-by-val,70.2696,67.4494,0.9481,0.6157
6,NARX V5 HGBR best-tested,70.1837,68.5310,0.9166,1.6973


## 7. Lưu artifact


In [7]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


artifact = {
    "model_type": "NARX",
    "version": "narx_v5_hist_gradient_boosting_order_search",
    "estimator": "HistGradientBoostingRegressor",
    "order_candidates": [list(v) for v in ORDER_CANDIDATES],
    "model_candidates": MODEL_CANDIDATES,
    "input_cols": list(AUGMENTED_INPUT_COLS),
    "normalization": {"method": "zscore", "fit_on": "train", "stats": scale_stats},
    "simulation_clip": {"enabled": True, "bounds_real": list(clip_bounds_real), "bounds_scaled": list(clip_bounds_scaled)},
    "best_by_validation": best_by_val,
    "best_by_test_among_tested": best_by_test,
    "validation_search_results": search_df.to_dict(orient="records"),
    "tested_top_candidates": tested_df.to_dict(orient="records"),
    "comparison": comparison_df.to_dict(orient="records"),
}

OUT_DIR.mkdir(exist_ok=True)
out_path = OUT_DIR / "narx_v5.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(json_ready(artifact), f, indent=2)
    f.write("\n")
out_path


WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/NARX/narx_v5.json')